# 01. Variant distribution across genomic windows

Describe the chromosomal distribution of genetic variation using window-level variant counts and allele frequencies.


# 01. 基因组窗口中的变异分布

利用窗口水平的变异数量和等位基因频率，描述遗传变异沿染色体的分布。


## 1. Variant data


## 1. 变异数据


### 1.1 Analysis setup


### 1.1 分析准备


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import socket
import sys
from datetime import datetime
from time import perf_counter
import pandas as pd

%load_ext autoreload
%autoreload 2

project_dir = '/absolute/remote/project/root'  # Replace with the confirmed config path.
os.chdir(project_dir)
project_dir = os.getcwd()
data_dir = f'{project_dir}/data'
source_dir = f'{data_dir}/source'
results_dir = f'{data_dir}/results'

os.makedirs(data_dir, exist_ok=True)
os.makedirs(source_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from src.plot_style import set_plot_style
set_plot_style()

print(f'Host: {socket.gethostname()}')
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.executable}')
print(f"Environment prefix: {os.environ.get('CONDA_PREFIX', '<not set>')}")

### 1.2 Variant coordinates and allele frequencies

Use variant coordinates and minor allele frequencies (MAF) to describe genetic variation.


### 1.2 变异坐标与等位基因频率

利用变异坐标和次要等位基因频率（MAF）描述遗传变异。


In [ ]:
variant_file = f'{source_dir}/variants.csv'
variant_table_h5 = f'{results_dir}/variant_table.h5'
start_time = perf_counter()

if os.path.exists(variant_table_h5):
    variants_df = pd.read_hdf(variant_table_h5, key='df')
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {variant_table_h5}')
else:
    variants_df = pd.read_csv(variant_file, dtype={'CHROM': str}, usecols=['CHROM', 'POS', 'MAF'])
    temporary_file = f'{variant_table_h5}.tmp'
    variants_df.to_hdf(temporary_file, key='df', mode='w')
    os.replace(temporary_file, variant_table_h5)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {variant_table_h5}')

print(f'Variants: {len(variants_df):,}; elapsed: {perf_counter() - start_time:.1f}s')
variants_df.head()

## 2. Window-level variant summaries

Summarize variant counts and mean MAF in genomic windows to describe their chromosomal distribution.


## 2. 窗口水平的变异汇总

在基因组窗口中汇总变异数量和平均 MAF，描述它们沿染色体的分布。


### 2.1 Window statistics


### 2.1 窗口统计


In [ ]:
window_size = 500
min_maf = 0.05
summary_file = f'{results_dir}/variant_windows.b{window_size}.maf{min_maf}.csv.gz'

- Summarize variants by chromosome and window


- 按染色体及窗口汇总变异


In [ ]:
from src.style_examples import build_chromosome_summaries

if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file, dtype={'CHROM': str})
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {summary_file}')
else:
    summary_df = build_chromosome_summaries(
        variants_df, window_size=window_size, min_maf=min_maf,
    )
    temporary_file = f'{summary_file}.tmp'
    summary_df.to_csv(temporary_file, index=False, compression='gzip')
    os.replace(temporary_file, summary_file)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {summary_file}')

print(f'Occupied windows: {len(summary_df):,}; columns: {summary_df.columns.tolist()}')
summary_df.head()

## 3. Chromosome profiles

Compare the mean MAF of retained variants across windows on selected chromosomes.


## 3. 染色体上的分布

比较选定染色体上不同窗口中保留变异的平均 MAF。


In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from src.style_examples import plot_window_maf

summary_df = pd.read_csv(summary_file, dtype={'CHROM': str})
chromosomes = ['Chr1', 'Chr2']  # Match identifiers in the input table.
figure_prefix = f'{results_dir}/variant_windows.b{window_size}.maf{min_maf}.chromosome_profiles'
fig, axes = plt.subplots(1, len(chromosomes), figsize=(12, 3), squeeze=False)
plot_tables = []

for chromosome, ax in tqdm(list(zip(chromosomes, axes[0])), desc='Plotting profiles', unit='chromosome'):
    plot_window_maf(summary_df, chromosome, ax)
    ax.set_ylim(0, 0.5)
    ax.set_title(chromosome)
    plot_df = summary_df.loc[summary_df['CHROM'] == chromosome].sort_values('WINDOW_START').copy()
    plot_df['POSITION_MB'] = plot_df['WINDOW_START'] / 1e6
    plot_df['PANEL_CHROM'] = chromosome
    plot_tables.append(plot_df)

fig.tight_layout()
pd.concat(plot_tables, ignore_index=True).to_csv(f'{figure_prefix}.csv.gz', index=False, compression='gzip')
fig.savefig(f'{figure_prefix}.pdf')
fig.savefig(f'{figure_prefix}.png', dpi=200)
plt.show()
print(f'Figure and data: {figure_prefix}.{{pdf,png,csv.gz}}')


**Figure 1. Window-level allele-frequency profiles on selected chromosomes.** Each panel shows mean MAF among variants with MAF ≥0.05 in occupied 500 bp windows. The horizontal axis gives window starts in Mb. Lines connect occupied windows across gaps.


**图 1. 选定染色体上窗口水平的等位基因频率分布。** 各面板展示有变异的 500 bp 窗口中 MAF ≥0.05 变异的平均 MAF。横坐标表示窗口起点，单位为 Mb。折线跨越空缺连接有变异的窗口。
